In [4]:
import sys
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Permite importar a pasta src/ estando dentro de notebooks/
sys.path.append(os.path.abspath('..'))

from src.models import MLP
from src.train import treinar
from src.metrics import avaliar_regressao

def fixar_sementes(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fixar_sementes(42)
print("PyTorch versão:", torch.__version__)

PyTorch versão: 2.15.0+cpu


In [5]:
# 1. Carregar os dados divididos na atividade anterior
# Substitua 'target_column' pelo nome real da sua coluna alvo de regressão (ex: 'Mental_Wellbeing_Score')
target_col = 'Mental_Wellbeing_Score' 

df_train_full = pd.read_csv('../data/processed/train_regression.csv')
df_test = pd.read_csv('../data/processed/test_regression.csv')

X_full = df_train_full.drop(columns=[target_col]).values
y_full = df_train_full[target_col].values

X_test = df_test.drop(columns=[target_col]).values
y_test = df_test[target_col].values
features_names = df_train_full.drop(columns=[target_col]).columns.tolist()

# 2. Separar validação (20% do treino)
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

# 3. Padronização (Ajustar scaler APENAS no X_train e y_train)
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_val_sc = scaler_X.transform(X_val)
X_test_sc = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_val_sc = scaler_y.transform(y_val.reshape(-1, 1)).flatten()
y_test_sc = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# 4. Converter para Tensores (float32 exigido no PDF)
def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_tr_t, y_tr_t = to_tensor(X_train_sc, y_train_sc)
X_va_t, y_va_t = to_tensor(X_val_sc, y_val_sc)
X_te_t, y_te_t = to_tensor(X_test_sc, y_test_sc)

# 5. DataLoaders
batch_size = 32
dl_train = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
dl_val = DataLoader(TensorDataset(X_va_t, y_va_t), batch_size=batch_size, shuffle=False)
dl_test = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=batch_size, shuffle=False)

print(f"Treino: {X_train.shape[0]} | Validação: {X_val.shape[0]} | Teste: {X_test.shape[0]}")

Treino: 960 | Validação: 240 | Teste: 300


In [6]:
# Hiperparâmetros Baseline
config_baseline = {
    "in_dim": X_train.shape[1],
    "out_dim": 1,
    "hidden": [32, 32],
    "activation": "ReLU",
    "dropout": 0.0
}

lr = 1e-3
epocas = 200
paciencia = 20
loss_fn = nn.MSELoss()

sementes = [42, 100, 2024, 7, 999]
historicos = []
modelos_treinados = []

for semente in sementes:
    fixar_sementes(semente)
    modelo = MLP(**config_baseline)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    
    print(f"Treinando semente {semente}...")
    hist = treinar(modelo, dl_train, dl_val, loss_fn, otimizador, n_epocas=epocas, paciencia=paciencia)
    
    historicos.append(hist)
    modelos_treinados.append(modelo)
    
    melhor_epoca = len(hist["val"]) - paciencia
    melhor_val = min(hist["val"])
    print(f" -> Parou na época {len(hist['val'])} (Melhor Val Loss: {melhor_val:.4f} na época {melhor_epoca})")

# Seleciona o modelo com o menor erro de validação entre as sementes
melhor_idx = np.argmin([min(h["val"]) for h in historicos])
modelo_vencedor = modelos_treinados[melhor_idx]
print(f"\nModelo escolhido para salvar: Semente {sementes[melhor_idx]}")

Treinando semente 42...
 -> Parou na época 32 (Melhor Val Loss: 0.3399 na época 12)
Treinando semente 100...
 -> Parou na época 65 (Melhor Val Loss: 0.3116 na época 45)
Treinando semente 2024...
 -> Parou na época 56 (Melhor Val Loss: 0.2969 na época 36)
Treinando semente 7...
 -> Parou na época 55 (Melhor Val Loss: 0.2898 na época 35)
Treinando semente 999...
 -> Parou na época 60 (Melhor Val Loss: 0.2712 na época 40)

Modelo escolhido para salvar: Semente 999


In [7]:
os.makedirs('../models', exist_ok=True)
caminho_modelo = '../models/modelo_regressao.pt'

checkpoint = {
    "task": "regression",
    "state_dict": modelo_vencedor.state_dict(),
    "config": config_baseline,
    "features": features_names,
    # Salvando os parâmetros do StandardScaler (X)
    "x_media": torch.tensor(scaler_X.mean_, dtype=torch.float32),
    "x_desvio": torch.tensor(scaler_X.scale_, dtype=torch.float32),
    # Salvando os parâmetros do StandardScaler (Y - Regressão)
    "y_media": scaler_y.mean_[0],
    "y_desvio": scaler_y.scale_[0]
}

torch.save(checkpoint, caminho_modelo)
print(f"Checkpoint completo salvo em: {caminho_modelo}")

Checkpoint completo salvo em: ../models/modelo_regressao.pt


In [8]:
# Coloca o modelo em modo de avaliação e prevê o conjunto de teste
modelo_vencedor.eval()

with torch.inference_mode():
    saidas_sc = modelo_vencedor(X_te_t)
    # Reverte a padronização do alvo (multiplica pelo desvio e soma a média)
    preds_original_scale = (saidas_sc.numpy() * scaler_y.scale_[0]) + scaler_y.mean_[0]

metricas = avaliar_regressao(y_test, preds_original_scale)

print("Métricas no Conjunto de Teste (Escala Original):")
for k, v in metricas.items():
    print(f"  {k}: {v:.4f}")

Métricas no Conjunto de Teste (Escala Original):
  MAE: 0.4271
  MSE: 0.2921
  RMSE: 0.5405
  MAPE: 1.3978
  R2: 0.7278
  MedAE: 0.3301
